In [1]:
BASE_MDOEL: str = "jhu-clsp/mmBERT-small"

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MDOEL)
tokenizer.add_special_tokens({"additional_special_tokens": ["<ent>"]})


/home/cheong/Desktop/Work/glinerminus/.venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


1

In [3]:
from transformers import AutoModel

model = AutoModel.from_pretrained(BASE_MDOEL)
model.resize_token_embeddings(len(tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(256001, 384, padding_idx=0)

In [ ]:
inputs = tokenizer(
    ["hellow this is a test sentence", "And this is the second sentence"],
    return_offsets_mapping=True,
)
inputs.offset_mapping

[[(0, 0),
  (0, 2),
  (2, 6),
  (6, 11),
  (11, 14),
  (14, 16),
  (16, 21),
  (21, 30),
  (0, 0)],
 [(0, 0), (0, 3), (3, 8), (8, 11), (11, 15), (15, 22), (22, 31), (0, 0)]]

In [ ]:
from typing import Any

import torch
from transformers.tokenization_utils_base import BatchEncoding
from transformers.tokenization_utils_fast import PreTrainedTokenizerFast


def tokenize(
    entities: list[list[str]],
    texts: list[str],
    spans: list[list[tuple[int, int]]],
    ent_token: str,
    tokenizer: PreTrainedTokenizerFast,
    **kwargs: dict[str, Any],
) -> BatchEncoding:
    """Tokenizes entities and text.

    Args:
        entities (list[str]): List of entities
        texts (list[str]): Text to tokenize
        spans (list[list[tuple[int, int]]]): List of spans
        ent_token (str): Special token for entities.
        tokenizer (PreTrainedTokenizerFast): Tokenizer to use
        **kwargs: Additional keyword arguments for the tokenizer

    Returns:
        BatchEncoding

    Raises:
        ValueError: If entities and text have different lengths
        TypeError: If input types are invalid

    """

    def make_input_str(entity: str, text: str) -> str:
        return f"{ent_token}{ent_token.join(entity)}{tokenizer.sep_token}{text}"

    if isinstance(entities, list) and isinstance(entities[0], list):
        if len(entities) != len(texts):
            msg = f"Entities length ({len(entities)}) must match text length ({len(texts)})"
            raise ValueError(msg)
        input_strs: list[str] = [make_input_str(e, t) for e, t in zip(entities, texts, strict=True)]
    else:
        msg = f"Invalid input types: {type(entities)} and {type(texts)}"
        raise TypeError(msg)
    if "return_offsets_mapping" in kwargs:
        kwargs.pop("return_offsets_mapping")
    if "return_tensors" in kwargs:
        kwargs.pop("return_tensors")
    inputs: BatchEncoding = tokenizer(
        input_strs,
        return_tensors="pt",
        return_offsets_mapping=True,
        **kwargs,
    )
    ent_token_id: int = tokenizer.all_special_ids[-1]
    rows, cols = torch.nonzero(inputs.input_ids == ent_token_id, as_tuple=True)
    inputs["entity_index"] = [[] for _ in range(len(texts))]
    for row, col in zip(rows.tolist(), cols.tolist(), strict=True):
        inputs["entity_index"][row].append(col)
    inputs["spans"] = [[] for _ in range(len(texts))]
    for i, seq_spans in enumerate(spans):
        text = texts[i]
        input_str = input_strs[i]
        offset = len(input_str) - len(text)
        for start_, end_ in seq_spans:
            start = start_ + offset
            end = end_ + offset
            start_token_idx = None
            end_token_idx = None
            for j, (tid, (s, e)) in enumerate(zip(inputs.input_ids[i], inputs.offset_mapping[i])):
                if tid.item() in tokenizer.all_special_ids:
                    continue
                if start_token_idx is not None and end_token_idx is not None:
                    break
                if s <= start < e:
                    start_token_idx = j
                if s < end <= e:
                    end_token_idx = j
            inputs["spans"][i].append((start_token_idx, end_token_idx))
    inputs.pop("offset_mapping")
    return inputs


inputs = tokenize(
    [["person"], ["organization"]],
    ["John Doe is a person and Google is an organization.", "Tina is a person"],
    spans=[[(0, 8), (25, 31)], [(0, 4)]],
    ent_token="<ent>",
    tokenizer=tokenizer,
    padding=True,
    truncation=True,
)
inputs

{'input_ids': tensor([[     2, 256000,   1552,      1,   3350,  70229,    603,    476,   1552,
            578,   6238,    603,    671,   8344, 235265,      1],
        [     2, 256000,   8344,      1,  57100,    603,    476,   1552,      1,
              0,      0,      0,      0,      0,      0,      0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]]), 'entity_index': [[1], [1]], 'spans': [[(4, 5), (10, 10)], [(4, 4)]]}

In [ ]:
features = model(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask)

In [ ]:
entity_matrix = torch.stack(
    [features.last_hidden_state[i, j] for i, item in enumerate(inputs.entity_index) for j in item],
)
entity_matrix.shape

torch.Size([2, 384])

In [ ]:
span_features = torch.stack(
    [
        features.last_hidden_state[b, span[0] : span[1] + 1].sum(dim=0)
        for b, spans in enumerate(inputs.spans)
        for span in spans
    ],
)
span_features.shape

torch.Size([3, 384])

In [83]:
import torch
from torch import nn

W = nn.Parameter(torch.empty(3, 384, 384))
for k in range(W.size(0)):
    nn.init.xavier_uniform_(W[k])

lne = nn.LayerNorm(384)
lns = nn.LayerNorm(384)

result = torch.einsum("qa,kab,jb->qjk", lne(entity_matrix), W, lns(span_features))
result

tensor([[[20.2734,  5.8984, 15.7223],
         [21.6094,  7.1973, 17.5237],
         [11.0603, 24.8766,  5.7673]],

        [[19.5389,  0.6697, 22.6465],
         [22.1829,  0.3071, 22.2362],
         [20.8050, 22.4660, 10.2669]]], grad_fn=<ViewBackward0>)